## This notebook lets you create your neo4j graph and vector database and add to it

### The same code works for already created graphs and databases. Instead of creating new ones running the code will connect to your exiting database if it already exists at the location given in the .env file

## Neo4j Setup (Skip if already done)

#### 1. Create a free instance at [neo4j.com/aura](https://neo4j.com/cloud/platform/aura-graph-database/) — save the password, it's only shown once. This will also download credentials make sure to copy these ones into your .env

#### 2. Create a `.env` file in your project root:
```env
NEO4J_URI      = neo4j+s://<YOUR_INSTANCE_ID>.databases.neo4j.io
NEO4J_USERNAME = neo4j
NEO4J_PASSWORD = <YOUR_PASSWORD>
NEO4J_DATABASE = <YOUR_INSTANCE_ID>
```

## Import Statements

In [1]:
import os
import glob
import numpy as np
from typing import List, Dict
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings.base import Embeddings
from langchain_community.vectorstores import Neo4jVector
from langchain_community.graphs import Neo4jGraph
from dotenv import load_dotenv
from neo4j import GraphDatabase

/Users/Aryan/miniconda3/envs/RAG_Langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup Configuration

In [2]:
load_dotenv(dotenv_path="../.env", override=True)
NEO4J_URI = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")


#TODO make INDEX/CHUNK name not hardcoded but a function
#NOTE: what these are set to is what is used to instantiate an agent. It is how it knows where in the graph to look 
#Ex: agent = personAgent("essay_chunk_agentspace", "chunk_index")
INDEX_NAME = "essay_chunk_agentspace"
CHUNK_NAME = "chunk_index"

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

## Embeddings Functions

In [3]:
class GemmaEmbeddings(Embeddings):
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def _normalize(self, v):
        norm = np.linalg.norm(v)
        return v / norm if norm > 0 else v

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        vectors = self.model.encode(texts, convert_to_numpy=True)
        return [self._normalize(v).tolist() for v in vectors]

    def embed_query(self, text: str) -> List[float]:
        v = self.model.encode([text], convert_to_numpy=True)[0]
        return self._normalize(v).tolist()

## Data Ingestion Functions

In [4]:
def load_pdfs(path_glob: str):
    docs = []
    for file in glob.glob(path_glob):
        loader = PyMuPDFLoader(
            file)
        docs.extend(loader.load())
    return docs


def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP, 
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = splitter.split_documents(docs)

    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
    return chunks

## Creating Graph Functions

### Connecting to neo4j driver

In [5]:
def get_driver():
    """Get Neo4j driver - create once and reuse."""
    return GraphDatabase.driver(
        NEO4J_URI, 
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
    )

### Create/Get VectorStore passing the driver around 

In [6]:
def get_or_create_vectorstore(embeddings, driver):
    """
    Connect to existing vector store OR create empty one.
    """
    

    with driver.session() as session:
        result = session.run("""
            SHOW INDEXES
            YIELD name, type
            WHERE name = $index_name AND type = 'VECTOR'
            RETURN count(*) > 0 as exists
        """, index_name=INDEX_NAME)
        
        record = result.single()
        index_exists = record['exists'] if record else False

    
    if index_exists:
        print(f"✅ Connecting to existing vector index: {INDEX_NAME}")
        vectorstore = Neo4jVector(
            embedding=embeddings,
            url=NEO4J_URI,
            username=NEO4J_USERNAME,
            password=NEO4J_PASSWORD,
            index_name=INDEX_NAME,
            node_label="Chunk",
            text_node_property="text",
            embedding_node_property="embedding",
        )
    else:
        print(f"🆕 Creating new vector index: {INDEX_NAME}")
        vectorstore = Neo4jVector.from_documents(
            documents=[],  # Empty - just creates index
            embedding=embeddings,
            url=NEO4J_URI,
            username=NEO4J_USERNAME,
            password=NEO4J_PASSWORD,
            index_name=INDEX_NAME,
            node_label="Chunk",
            text_node_property="text",
            embedding_node_property="embedding",
        )
        print(f"✅ Created empty vector store")
    
    return vectorstore

### Add documents to vector store and graph functions

In [7]:
def build_graph_relationships(chunks, driver):
    """Build Document nodes and relationships."""

    by_source: Dict[str, List] = {}
    for c in chunks:
        src = c.metadata.get("source", "unknown")
        by_source.setdefault(src, []).append(c)

    with driver.session() as session:
        session.run("""
            CREATE FULLTEXT INDEX chunk_index IF NOT EXISTS
            FOR (n:Chunk) ON EACH [n.text]
        """)
        
        for source, source_chunks in by_source.items():
            source_chunks.sort(key=lambda x: x.metadata["chunk_id"])

            session.run(
                "MERGE (d:Document {name: $name}) SET d.chunk_count = $n",
                {"name": source, "n": len(source_chunks)}
            )

            for i, c in enumerate(source_chunks):
                session.run(
                    """
                    MATCH (d:Document {name: $source})
                    MATCH (c:Chunk {chunk_id: $cid})
                    MERGE (c)-[:PART_OF]->(d)
                    """,
                    {"source": source, "cid": c.metadata["chunk_id"]}
                )

                if i < len(source_chunks) - 1:
                    session.run(
                        """
                        MATCH (c1:Chunk {chunk_id: $c1})
                        MATCH (c2:Chunk {chunk_id: $c2})
                        MERGE (c1)-[:NEXT]->(c2)
                        """,
                        {
                            "c1": c.metadata["chunk_id"],
                            "c2": source_chunks[i + 1].metadata["chunk_id"],
                        },
                    )



def add_documents(path, vectorstore, driver):
    """"Function where you can take file path and get back vector embeddings"""
    pdfs = load_pdfs(path)
    chunks = split_documents(pdfs)
    vectorstore.add_documents(chunks)
    build_graph_relationships(chunks, driver)


## Do the actual data adding

### Connect to everything that is needed

In [8]:
#connect to everything
driver = get_driver()
embeddings = GemmaEmbeddings(EMBEDDING_MODEL_NAME)
vectorstore = get_or_create_vectorstore(embeddings, driver)

🆕 Creating new vector index: essay_chunk_agentspace
✅ Created empty vector store


### BodyParagraphs.pdf in this folder is a good starter

In [ ]:
#if not already in the vectorstore add in my documents 
Data = "Knowledge/BodyParagraphs.pdf"
add_documents(Data, vectorstore, driver)